In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("funnel_events_sample.csv")
df

,user_id,step,timestamp
0,U173,visited_site,2026-07-20 04:01:00
1,U041,details_filled,2026-07-20 04:18:00
2,U165,visited_site,2026-07-20 03:53:00
3,U132,visited_site,2026-07-20 03:58:00
4,U164,visited_site,2026-07-20 04:08:00
...,...,...,...
537,U169,details_filled,2026-07-20 04:29:00
538,U126,signup_started,2026-07-20 03:58:00
539,U200,details_filled,2026-07-20 04:11:00
540,U070,visited_site,2026-07-20 03:45:00


In [3]:
df.head(10)

,user_id,step,timestamp
0,U173,visited_site,2026-07-20 04:01:00
1,U041,details_filled,2026-07-20 04:18:00
2,U165,visited_site,2026-07-20 03:53:00
3,U132,visited_site,2026-07-20 03:58:00
4,U164,visited_site,2026-07-20 04:08:00
5,U152,visited_site,2026-07-20 03:50:00
6,U072,visited_site,2026-07-20 03:49:00
7,U075,signup_started,2026-07-20 04:20:00
8,U097,signup_started,2026-07-20 04:09:00
9,U024,details_filled,2026-07-20 04:02:00


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542 entries, 0 to 541
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   user_id    542 non-null    object        
 1   step       542 non-null    object        
 2   timestamp  542 non-null    datetime64[ns]
dtypes: datetime64[ns](1), object(2)
memory usage: 12.8+ KB


In [8]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [5]:
df.describe()

,user_id,step,timestamp
count,542,542,542
unique,200,5,94
top,U041,visited_site,2026-07-20 04:09:00
freq,3,200,17


In [6]:
df.isnull().sum()

user_id      0
step         0
timestamp    0
dtype: int64

In [7]:
df["step"].unique()

array(['visited_site', 'details_filled', 'signup_started',
       'purchase_completed', 'email_verified'], dtype=object)

In [10]:
df = df.drop_duplicates(subset=["user_id","step"])

In [11]:
df

,user_id,step,timestamp
0,U173,visited_site,2026-07-20 04:01:00
1,U041,details_filled,2026-07-20 04:18:00
2,U165,visited_site,2026-07-20 03:53:00
3,U132,visited_site,2026-07-20 03:58:00
4,U164,visited_site,2026-07-20 04:08:00
...,...,...,...
537,U169,details_filled,2026-07-20 04:29:00
538,U126,signup_started,2026-07-20 03:58:00
539,U200,details_filled,2026-07-20 04:11:00
540,U070,visited_site,2026-07-20 03:45:00


In [14]:
steps = [
    "visited_site",
    "signup_started",
    "details_filled",
    "email_verified",
    "purchase_completed"
]

In [16]:
# User Counts on respective Stages

user_counts = (df.groupby("step")["user_id"].nunique().reindex(steps))
user_counts

step
visited_site          200
signup_started        150
details_filled         96
email_verified         52
purchase_completed     44
Name: user_id, dtype: int64

In [18]:
# Funnel Table 

summary = pd.DataFrame({
    "Stage": steps,
    "Users": user_counts.values
})
summary

,Stage,Users
0,visited_site,200
1,signup_started,150
2,details_filled,96
3,email_verified,52
4,purchase_completed,44


In [22]:
# Conversion Rate 

conversion_rate = (user_counts / user_counts.shift(1)) * 100
conversion_rate.iloc[0] = 100
conversion_rate = conversion_rate.round(2)

conversion_rate

step
visited_site          100.00
signup_started         75.00
details_filled         64.00
email_verified         54.17
purchase_completed     84.62
Name: user_id, dtype: float64

In [23]:
# Drop-Off

drop_off = user_counts.shift(1) - user_counts
drop_off.iloc[0] = 0
drop_off = drop_off.astype(int)

drop_off

step
visited_site           0
signup_started        50
details_filled        54
email_verified        44
purchase_completed     8
Name: user_id, dtype: int64

In [35]:
# Biggest Drop-Off

biggest_drop_stage = drop_off.idxmax()
print(f"{biggest_drop_stage}, Users_count{ -drop_off.max()}")


details_filled, Users_count-54


In [39]:
final_summary = pd.DataFrame({
    "Stage": user_counts.index.str.replace("_", " ").str.title(),
    "Users": user_counts.values,
    "Conversion Rate": conversion_rate.values,
    "Drop-off": drop_off.values
})

final_summary.to_csv("funnel_summary.csv", index=False)



# Recommendation

In [ ]:
The largest drop-off occurs between Signup Started and Details Filled, where 54 users leave the funnel. 
This indicates that the registration form may be too lengthy, confusing, or require unnecessary information.
To improve conversion, consider simplifying the form by reducing mandatory fields or direct google sign-in option.